In [2]:
import os
os.chdir(r"C:\Users\Noel\Documents\Personal Projects\f1-rag-knowledge-assistant")
print(os.getcwd())

C:\Users\Noel\Documents\Personal Projects\f1-rag-knowledge-assistant


In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.llms import Ollama
from langchain.chains import RetrievalQA

In [4]:
# Define the path to your F1 PDF
PDF_PATH = "C:/Users/Noel/Documents/Personal Projects/f1-rag-knowledge-assistant/data/F1_2024_BritishGP_RaceReport.pdf"

# Create the loader
loader = PyPDFLoader(PDF_PATH)

# Load the document pages
pages = loader.load()

# See what we got
print(f"Total pages loaded: {len(pages)}")
print(f"\nFirst page preview:\n{pages[0].page_content[:500]}")

Total pages loaded: 7

First page preview:
FIA FORMULA ONE WORLD CHAMPIONSHIP
 
 2024 FORMULA 1
QATAR AIRWAYS
BRITISH GRAND PRIX
 OFFICIAL RACE REPORT & STATISTICAL REVIEW
 
Circuit
Silverstone Circuit, Northamptonshire, United Kingdom
Race Date
Sunday, 7 July 2024
Round
Round 12 of 24 — 2024 FIA Formula One World Championship
Distance
52 laps × 5.891 km = 306.198 km
Weather
Partly cloudy, 21°C ambient, 38°C track surface
Attendance
480,000 (race weekend total — record for Silverstone)
Document Ref
F1-2024-GBR-RACE-REPORT-v1.2
This docum


In [5]:
#Create the splitters
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200
)

#Split the pages into chunks
chunks = splitter.split_documents(pages)

print(f"total chunks created :{len(chunks)}")
print(f"\nFirst chunk preview:\n{chunks[0].page_content}")

total chunks created :16

First chunk preview:
FIA FORMULA ONE WORLD CHAMPIONSHIP
 
 2024 FORMULA 1
QATAR AIRWAYS
BRITISH GRAND PRIX
 OFFICIAL RACE REPORT & STATISTICAL REVIEW
 
Circuit
Silverstone Circuit, Northamptonshire, United Kingdom
Race Date
Sunday, 7 July 2024
Round
Round 12 of 24 — 2024 FIA Formula One World Championship
Distance
52 laps × 5.891 km = 306.198 km
Weather
Partly cloudy, 21°C ambient, 38°C track surface
Attendance
480,000 (race weekend total — record for Silverstone)
Document Ref
F1-2024-GBR-RACE-REPORT-v1.2
This document is published by the FIA Formula One Administration and is intended for official championship and media
purposes. All lap time data sourced from official FIA transponder systems. Tyre compound information provided by Pirelli
Motorsport.


In [6]:
import shutil
if os.path.exists("chromadb"):
    shutil.rmtree("chromadb")

    
#Load the embedding model
embedding_model = HuggingFaceEmbeddings(
    model_name = "all-MiniLM-L6-v2"
)

#create ChromaDB and store chunks
vector_store = Chroma.from_documents(
    documents = chunks,
    embedding = embedding_model,
    persist_directory = "chroma_db"
)

print(f"Total vectors stored :{vector_store._collection.count()}")


C:\Users\Noel\AppData\Local\Temp\ipykernel_23076\3420019848.py:7: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 0.3.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  embedding_model = HuggingFaceEmbeddings(
c:\Users\Noel\Documents\Personal Projects\f1-rag-knowledge-assistant\venv\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument b

Total vectors stored :16


In [7]:
#Load the LLM
llm = Ollama(model="llama3.2")

#Build the RAG Chain
qa_chain = RetrievalQA.from_chain_type(
    llm = llm,
    retriever = vector_store.as_retriever(search_kwargs ={"k":5}),
    chain_type = "stuff"
)

print("RAG chain ready!")

RAG chain ready!


In [8]:
#Ask the question
question = "Who won the 2024 British Grand Prix and by how much?"
result = qa_chain.invoke({"query":question})
print(result["result"])

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


I don't know who won the 2024 British Grand Prix, but I do know that Lando Norris secured pole position in a dramatic qualifying session on Saturday, posting a lap of 1:25.912.


In [9]:
# See exactly what chunks the chain is sending to the LLM
question = "Who won the 2024 British Grand Prix and by how much?"

docs = vector_store.similarity_search(question, k=5)

for i, doc in enumerate(docs):
    print(f"\n--- Chunk {i+1} ---")
    print(doc.page_content[:300])


--- Chunk 1 ---
calendar, having hosted the inaugural World Championship round in 1950. The 5.891 km layout features 18
corners including the legendary high-speed Copse, Maggotts, and Becketts complexes, where drivers
experience lateral g-forces in excess of 5g. The circuit was resurfaced in Sectors 2 and 3 in Nove

--- Chunk 2 ---
1. RACE NARRATIVE & MATCH REPORT
1.1 Pre-Race Grid & Starting Conditions
The 2024 British Grand Prix commenced under overcast skies at Silverstone, with ambient temperatures of
21°C and a track surface temperature of 38°C — considerably cooler than forecast and a significant factor in
tyre degradati

--- Chunk 3 ---
FIA FORMULA ONE WORLD CHAMPIONSHIP
 
 2024 FORMULA 1
QATAR AIRWAYS
BRITISH GRAND PRIX
 OFFICIAL RACE REPORT & STATISTICAL REVIEW
 
Circuit
Silverstone Circuit, Northamptonshire, United Kingdom
Race Date
Sunday, 7 July 2024
Round
Round 12 of 24 — 2024 FIA Formula One World Championship
Distance
52 la

--- Chunk 4 ---
4. CHAMPIONSHIP STANDINGS — A

In [10]:
print(f"Vectors in DB: {vector_store._collection.count()}")

# Also print ALL chunk contents to verify no duplicates
all_docs = vector_store.similarity_search("Norris victory", k=20)
print(f"Total unique chunks retrieved: {len(all_docs)}")

Number of requested results 20 is greater than number of elements in index 16, updating n_results = 16


Vectors in DB: 16
Total unique chunks retrieved: 16


In [11]:
# Try a more specific question
question2 = "Which driver crossed the finish line first at Silverstone in 2024?"
result2 = qa_chain.invoke({"query": question2})
print(result2["result"])

Lando Norris.


In [12]:
# Try another angle
question3 = "What happened on the final lap of the 2024 British Grand Prix?"
result3 = qa_chain.invoke({"query": question3})
print(result3["result"])

I don't know what happened on the final lap of the 2024 British Grand Prix. The provided context only mentions that the race result was declared final by the Clerk of the Course at 17:34 local time, but it does not provide information about the events that occurred during the final lap.
